# Lidar autoencoder - harom architektura osszehasonlitasa

Harom modell tanul UGYANAZON az adaton. Mindharom TERBELI jellemzoterkepet
ad (nem lapitott vektort) - az RL sajat CNN feature-extractora dolgozza fel:

| modell | bemenet | mit tanul |
|---|---|---|
| `bev_ae` | nyers pontfelho | fix BEV statisztika (occupancy + z_max/z_min) -> conv AE |
| `graph_ae` | range image | dinamikus graf-konvolucio (EdgeConv) |
| `point_mae` | nyers pontfelho | maszkolt patch-rekonstrukcio (Transformer) |

## 1. Modulok, beallitasok

In [ ]:
import gc
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from lidar.point_mae import PointMAE, _chamfer_per_sample
from lidar.bev_ae import BEVConvAE
from lidar.graph_ae import LidarAE, DDCONFIG, points_to_range_image

DATA_DIR = "dataset/lidar"
EPOCHS = 40
N_FRAMES = None           # None = minden frame

N_POINTS = 24576

# A point_mae batch-e 128-rol 64-re ment. Ok: a num_group 128 -> 256 es a
# group_size 16 -> 32 (lasd a 6. szakaszt), ami a group() cdist-jet es a
# Chamfer tavolsagmatrixat is megnoveli. MERVE, ezen a 8.3 GB-os GPU-n:
#
#     batch 32 -> 5.9 GB    batch 64 -> 6.0 GB    batch 96 -> OOM
BATCH = {"bev_ae": 768, "graph_ae": 32, "point_mae": 100}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
def free_vram(*objects):
    for o in objects:
        if hasattr(o, "cpu"):
            o.cpu()
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def vram():
    """Aktualis GPU memoriahasznalat."""
    if not torch.cuda.is_available():
        return "n/a"
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    return (f"{torch.cuda.memory_allocated() / 1e9:.2f} GB foglalt / "
            f"{torch.cuda.memory_reserved() / 1e9:.2f} GB fenntartva / "
            f"{total:.1f} GB osszes")


print("indulaskor:", vram())

## 2. Adatok betoltese

**Harmas vagas (70 / 15 / 15), a szokasos szereposztassal:**

| halmaz | mire valo | ki latja |
|---|---|---|
| **train** | a sulyok tanulasa | a gradiens |
| **val** | early stopping, a legjobb epoch kivalasztasa | a DONTESEINK |
| **test** | a vegso, elfogulatlan meres | **csak egyszer, a legvegen** |

In [ ]:
PATHS = sorted(glob.glob(os.path.join(DATA_DIR, "*.npy")))
if N_FRAMES:
    PATHS = PATHS[:N_FRAMES]

# A keveres KOTELEZO: a framek idorendben keszultek, vagas elott a val/test
# a felvetel utolso szakasza lenne - mas utszakasz, mas forgalom, mint a train.
PERM = np.random.default_rng(0).permutation(len(PATHS))

VAL_FRAC, TEST_FRAC = 0.15, 0.15
N_TEST = int(len(PATHS) * TEST_FRAC)
N_VAL = int(len(PATHS) * VAL_FRAC)
N_TRAIN = len(PATHS) - N_VAL - N_TEST

print(f"{len(PATHS)} frame   train {N_TRAIN}   val {N_VAL}   test {N_TEST}")
print(f"RAM egy reprezentaciora: pontfelho "
      f"{len(PATHS) * N_POINTS * 3 * 4 / 1e9:.1f} GB, "
      f"range image {len(PATHS) * 64 * 512 * 4 / 1e9:.1f} GB")

In [ ]:
def _load(shape, fill, which=("train", "val"), workers=16):
    from concurrent.futures import ThreadPoolExecutor

    # A permutacio elso N_TEST eleme a test, utana N_VAL a val, a tobbi train.
    spans = {"test": (0, N_TEST),
             "val": (N_TEST, N_TEST + N_VAL),
             "train": (N_TEST + N_VAL, len(PATHS))}

    out = []
    for name in which:
        lo, hi = spans[name]
        idx = PERM[lo:hi]
        arr = np.empty((len(idx), *shape), dtype=np.float32)

        def read_one(k):
            arr[k] = fill(np.load(PATHS[idx[k]]), idx[k])

        with ThreadPoolExecutor(max_workers=workers) as ex:
            list(tqdm(ex.map(read_one, range(len(idx))), total=len(idx),
                      desc=f"{name:5s}", unit="frame"))
        out.append(torch.from_numpy(arr))
    return out


def load_clouds(which=("train", "val")):
    def fill(pts, idx):
        sel = np.random.default_rng(idx).choice(
            len(pts), N_POINTS, replace=len(pts) < N_POINTS)
        return pts[sel]
    return _load((N_POINTS, 3), fill, which)


def load_ranges(which=("train", "val")):
    # Az alakot a fuggvenytol kerdezzuk, nem beegetve: a TopoLiDM config
    # 64x1024-et ir elo, de ha ez valtozik, ez a cella nem szall el.
    shape = points_to_range_image(np.load(PATHS[0])).shape
    return _load(shape, lambda pts, idx: points_to_range_image(pts), which)


def drop(*names):
    """Tenzorok eldobasa a RAM-bol. A `del` utan a gc szabaditja fel."""
    for n in names:
        globals().pop(n, None)
    gc.collect()

### Mit tanulunk?

Harom veletlen frame, haromfele nezetben - pontosan az, amit a modellek
bemenetkent kapnak. Ehhez ideiglenesen betoltunk nehany frame-et.

In [ ]:
# Csak a megjelenitendo framek - nem a teljes adat.
SHOW_IDX = sorted(np.random.default_rng(1).choice(len(PATHS), 3, replace=False))
_show_raw = [np.load(PATHS[i]) for i in SHOW_IDX]
_show_pts = [torch.from_numpy(
    a[np.random.default_rng(i).choice(len(a), N_POINTS,
                                      replace=len(a) < N_POINTS)].astype(np.float32))
    for i, a in zip(SHOW_IDX, _show_raw)]

# A BEV statisztika fix keplet, nem tanult jellemzoterkep - egy friss
# modellpeldany is a vegleges kepet adja.
_probe = BEVConvAE().eval()
print(f"BEV racs {_probe.grid_h}x{_probe.grid_w}, "
      f"{_probe.hparams.voxel_size} m cella   |   framek: {SHOW_IDX}")

fig, axes = plt.subplots(3, 3, figsize=(17, 13))
for row, (idx, raw, pts) in enumerate(zip(SHOW_IDX, _show_raw, _show_pts)):
    frame = pts.numpy()

    ax = axes[row, 0]
    ax.scatter(frame[:, 0], frame[:, 1], c=frame[:, 2], s=0.3, cmap="viridis",
               vmin=-3, vmax=4, linewidths=0)
    ax.scatter([0], [0], c="red", s=40, marker="^")
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    ax.set_aspect("equal")
    ax.set_ylabel(f"frame {idx}\ny [m]")
    if row == 0:
        ax.set_title(f"pontfelho ({len(frame)} / {len(raw)} pont)\npoint_mae",
                     fontsize=11)

    ax = axes[row, 1]
    with torch.no_grad():
        occ = _probe.to_bev(pts.unsqueeze(0))[0, 0].numpy()
    _r = _probe.hparams.pc_range          # a modell sajat hatosugara
    ax.imshow(occ, cmap="gray_r", origin="lower", extent=[-_r, _r, -_r, _r])
    ax.set_xlabel("x [m]")
    if row == 0:
        ax.set_title("BEV occupancy (3 csatornabol)\nbev_ae", fontsize=11)
    ax.text(0.02, 0.96, f"{100 * occ.mean():.0f}% cella", transform=ax.transAxes,
            fontsize=9, va="top")

    ax = axes[row, 2]
    ri = points_to_range_image(raw)[0]
    ax.imshow(ri, cmap="magma", aspect="auto", vmin=-1, vmax=1)
    ax.set_xlabel("azimut (512 oszlop)")
    ax.set_ylabel("csatorna (64 sor)")
    if row == 0:
        ax.set_title("range image\ngraph_ae", fontsize=11)
    ax.text(0.02, 0.95, f"{100 * (ri > -0.999).mean():.0f}% kitoltott",
            transform=ax.transAxes, color="white", fontsize=9, va="top")

plt.tight_layout()
plt.show()
free_vram(_probe)
drop("_show_raw")

### Interaktiv 3D nezet

Forgathato, nagyithato. Ez a NYERS pontfelho — amibol mindharom modell
bemenete keszul.

In [ ]:
def show_3d(pts, title="", max_points=25000, size=1.2, extra=None,
            height=650):
    pts = pts.cpu().numpy() if hasattr(pts, "cpu") else np.asarray(pts)

    # Ritkitas, hogy a bongeszo ne akadjon meg.
    if len(pts) > max_points:
        sel = np.random.default_rng(0).choice(len(pts), max_points, replace=False)
        pts = pts[sel]

    traces = [go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="markers",
        marker=dict(size=size, color=pts[:, 2], colorscale="Viridis",
                    cmin=-3, cmax=4, opacity=0.85,
                    colorbar=dict(title="z [m]", thickness=12, len=0.6)),
        name=f"pontfelho ({len(pts)})",
        hovertemplate="x %{x:.1f}<br>y %{y:.1f}<br>z %{z:.1f}<extra></extra>",
    )]

    # Az auto helye: a lidar az origoban van (x=0, z=2.4 a szenzor magassaga,
    # de a pontok mar a szenzorhoz kepest vannak megadva).
    traces.append(go.Scatter3d(
        x=[0], y=[0], z=[0], mode="markers",
        marker=dict(size=6, color="red", symbol="diamond"),
        name="ego (lidar)", hoverinfo="name"))

    for e_pts, e_name, e_color, e_size in (extra or []):
        e_pts = e_pts.cpu().numpy() if hasattr(e_pts, "cpu") else np.asarray(e_pts)
        traces.append(go.Scatter3d(
            x=e_pts[:, 0], y=e_pts[:, 1], z=e_pts[:, 2], mode="markers",
            marker=dict(size=e_size, color=e_color), name=e_name))

    fig = go.Figure(traces)
    fig.update_layout(
        title=title,
        height=height,
        margin=dict(l=0, r=0, t=40 if title else 0, b=0),
        legend=dict(x=0, y=1),
        scene=dict(
            # aspectmode="data": a harom tengely ARANYA a valos meretekbol
            # jon. Enelkul a plotly kockara nyujtja a jelenetet, es a lapos
            # ut-sik magasnak tunne.
            aspectmode="data",
            xaxis_title="x [m]  (elore)",
            yaxis_title="y [m]  (balra)",
            zaxis_title="z [m]  (fel)",
            # Kezdo kameraallas: kicsit hatulrol-felulrol, mint egy
            # kovetokamera - igy egybol felismerheto a jelenet.
            camera=dict(eye=dict(x=-1.4, y=-1.4, z=0.9)),
        ),
    )
    fig.show()

In [ ]:
# Ugyanaz a harom frame 3D-ben, forgathatoan.
for idx, pts in zip(SHOW_IDX, _show_pts):
    show_3d(pts, title=f"frame {idx} - nyers pontfelho")

## 3. Kozos tanito fuggvenyek

A `train_model` **megszakithato es folytathato**: minden epoch vegen ment
(atomikusan, `.tmp` + atnevezes), es ujrainditaskor onnan folytatja, ahol
abbahagyta. Az optimizer allapota is mentodik — enelkul a folytatas nulla
lendulettel indulna, es az elso par epoch elrontana a tanulast.

In [ ]:
def evaluate(model, data, batch, parts=False):
    """Atlagos loss a teljes adathalmazon.

    parts=True : (loss, dict) - a modell sajat diagnosztikaja is, atlagolva.
        EZ FONTOS: mindharom modellnel elofordult, hogy a loss szepen csokkent,
        kozben a modell mast tanult meg, mint amit akartunk:

          bev_ae     a loss 0.021-en allt, de az occupancy IoU csak 0.58 volt
                     (precision 0.59 - felmillio cella tevesen foglalt)
          point_mae  a Chamfer 4.74, a trivialis patch-atlag 4.79 -> a modell
                     KOLLAPSZALT, minden patch-re kb. ugyanazt adta. Az OK a
                     normalizalas hianya volt (lasd a 6. szakaszt); a modell
                     most a `gain_pct` diagnosztikat is visszaadja, ami pont
                     ezt a viszonyt meri epochonkent.
          graph_ae   az L1 nagy reszet az URES teruletek tettek ki

        Egyetlen szambol ezek egyike sem latszik.
    """
    model.eval()
    total, n, acc = 0.0, 0, {}
    with torch.no_grad():
        for i in range(0, len(data), batch):
            xb = data[i:i + batch].to(DEVICE, non_blocking=True).float()
            if parts:
                loss, p = model.loss(xb, parts=True)
                for k, v in p.items():
                    acc[k] = acc.get(k, 0.0) + v * len(xb)
            else:
                loss = model.loss(xb)
            total += float(loss) * len(xb)
            n += len(xb)
    if not parts:
        return total / n
    return total / n, {k: v / n for k, v in acc.items()}


def train_model(model, label, batch, train_data, val_data,
                epochs=EPOCHS, lr=1e-3, patience=5, sched="plateau",
                resume=True, clip=5.0, warmup_frac=0.0):
    """`warmup_frac`: az ELSO epoch hany szazalekaban fusson fel az LR nullarol.

    TRANSFORMERNEL EZ KELL. A cosine/plateau scheduler epoch-szintu, vagyis az
    elso epoch TELJES egeszeben a maximalis LR-en fut - pont akkor, amikor a
    LayerNorm-ok, a mask_token es az attention sulyai meg semmilyen ertelmes
    allapotban nincsenek. A point_mae gradiens-normaja az elso lepeseknel
    merve 13-30 kozott van, a clip=5.0 pedig ezt levagja: a lepes IRANYA igy
    a veletlen kezdeti allapotbol jon, a HOSSZA meg maximalis.

    A warmup lepes-szintu (a batch-eken belul is valtozik), ezert nem lehet a
    meglevo epoch-szintu scheduler resze - a kettot egymas melle tesszuk: a
    warmup alatt a warmup hatarozza meg az LR-t, utana a scheduler.
    """
    model = model.to(DEVICE)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    if sched == "plateau":
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=2, min_lr=lr / 100)
    elif sched == "cosine":
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=epochs, eta_min=lr / 100)
    else:
        sched = None

    # Hany OPTIMIZER-LEPESEN at tart a felfutas (nem epoch, hanem lepes).
    steps_per_epoch = (len(train_data) + batch - 1) // batch
    warmup_steps = int(warmup_frac * steps_per_epoch)

    hist = {"label": label, "arch": type(model).__name__,
            "train": [], "val": [], "best": float("inf"),
            "params": sum(p.numel() for p in model.parameters()),
            "ckpt": f"lidar/{label}.ckpt"}
    best_state, bad, start_ep = None, 0, 1

    # --- FOLYTATAS a mentett checkpointbol -----------------------------
    if resume and os.path.exists(hist["ckpt"]):
        ck = torch.load(hist["ckpt"], map_location="cpu", weights_only=False)
        # Csak akkor folytatunk, ha UGYANARROL a modellrol van szo. Ha kozben
        # atirtad az architekturat, a sulyok alakja nem egyezne - olyankor
        # inkabb elolrol kezdunk, mint hogy hibaval elszalljon.
        if ck.get("arch") == hist["arch"] and "epoch" in ck:
            try:
                model.load_state_dict(ck["state_dict"])
                opt.load_state_dict(ck["optimizer"])
                if sched is not None and ck.get("sched") is not None:
                    sched.load_state_dict(ck["sched"])
                hist["train"] = list(ck.get("train", []))
                hist["val"] = list(ck.get("val", []))
                # A diag/lr listakat is vissza kell tolteni, kulonben a
                # folytatas utan rovidebbek lesznek, mint a train/val, es a
                # plot_history epoch-tengelye nem illeszkedik rajuk.
                hist["diag"] = list(ck.get("diag", []))
                hist["lr"] = list(ck.get("lr", []))
                hist["best"] = ck.get("best_val", float("inf"))
                best_state = ck.get("best_state", ck["state_dict"])
                bad = ck.get("bad", 0)
                start_ep = ck["epoch"] + 1
                if start_ep > epochs:
                    print(f"{label}: mar lefutott {ck['epoch']} epoch "
                          f"(best val {hist['best']:.5f}). Emeld az `epochs` "
                          f"erteket, vagy add at a resume=False-t az "
                          f"ujrakezdeshez.")
                    model.load_state_dict(best_state)
                    hist["epochs_run"] = len(hist["train"])
                    hist["time"] = 0.0
                    return hist, model
                print(f"{label}: FOLYTATAS a {ck['epoch']}. epoch utan "
                      f"(best val {hist['best']:.5f})")
            except (RuntimeError, KeyError) as e:
                print(f"{label}: a checkpoint nem illeszkedik ({type(e).__name__}), "
                      f"elolrol kezdjuk")
                best_state, bad, start_ep = None, 0, 1
                hist["train"], hist["val"], hist["best"] = [], [], float("inf")
                hist["diag"], hist["lr"] = [], []
        else:
            print(f"{label}: regi formatumu vagy mas architekturaju checkpoint, "
                  f"elolrol kezdjuk")

    print(f"{label}  ({hist['params'] / 1e6:.1f}M parameter, lr={lr}, "
          f"batch={batch}, {len(train_data)} train frame, "
          f"epoch {start_ep}-{epochs})")
    t0 = time.time()

    for ep in range(start_ep, epochs + 1):
        model.train()
        run, n = 0.0, 0
        perm = torch.randperm(len(train_data))
        bar = tqdm(range(0, len(train_data), batch), desc=f"epoch {ep}/{epochs}",
                   leave=False)
        for step, i in enumerate(bar):

            if warmup_steps and ep == 1 and step <= warmup_steps:
                w = min(step + 1, warmup_steps) / warmup_steps
                for g in opt.param_groups:
                    g["lr"] = lr * w
            xb = train_data[perm[i:i + batch]].to(DEVICE, non_blocking=True).float()
            loss = model.loss(xb)
            opt.zero_grad(set_to_none=True)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
            opt.step()
            run += float(loss.detach()) * len(xb)
            n += len(xb)
            bar.set_postfix(loss=f"{float(loss.detach()):.5f}")



        tr = run / n
        va, diag = evaluate(model, val_data, batch, parts=True)

        TRACK = {"LidarAE": "l1_occupied"}       # modell -> diagnosztika-kulcs
        key = TRACK.get(hist["arch"])
        score = diag[key] if key and key in diag else va

        if sched is not None:
            if isinstance(sched, torch.optim.lr_scheduler.ReduceLROnPlateau):
                sched.step(score)
            else:
                sched.step()
        hist["train"].append(tr)
        hist["val"].append(va)
        hist.setdefault("diag", []).append(diag)
        hist.setdefault("lr", []).append(opt.param_groups[0]["lr"])

        # Early stopping: ha `patience` epochon at nem javul, megallunk.
        improved = score < hist["best"]
        if improved:
            hist["best"], bad = score, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            bad += 1

        # MENTES MINDEN EPOCH VEGEN - ez teszi folytathatova a futast.
        # Eloszor egy .tmp fajlba irunk, aztan atnevezzuk: igy ha pont mentes
        # kozben szakad meg a futas, a regi checkpoint epen marad.
        tmp = hist["ckpt"] + ".tmp"
        torch.save({"state_dict": {k: v.detach().cpu()
                                   for k, v in model.state_dict().items()},
                    "best_state": best_state,
                    "optimizer": opt.state_dict(),
                    "sched": sched.state_dict() if sched is not None else None,
                    "epoch": ep,
                    "bad": bad,
                    "hparams": (dict(model.hparams) if hasattr(model, "hparams")
                                else {}),
                    "arch": hist["arch"],
                    "best_val": hist["best"],
                    "train": hist["train"], "val": hist["val"],
                    # A diagnosztika is menjen bele: a nyers loss onmagaban
                    # ertelmezhetetlen (mas patch-eles/sulyozas mellett mas a
                    # skalaja), a gain_pct viszont osszehasonlithato.
                    "diag": hist.get("diag", []),
                    "lr": hist.get("lr", [])}, tmp)
        os.replace(tmp, hist["ckpt"])

        # A diagnosztika a loss MELLE - abbol latszik, hogy a modell
        # tenyleg azt tanulja-e, amit akarunk (lasd evaluate docstring).
        extra = "  ".join(f"{k} {v:.3f}" for k, v in diag.items()
                          if k in ("occ_prec", "occ_rec", "z_mae_m", "spread_ratio",
                                   "l1_occupied", "l1_empty", "gain_pct",
                                   "cd_m2"))
        # Az aktualis LR is latszodjon - enelkul nem ellenorizheto, hogy a
        # scheduler tenyleg dolgozik-e.
        extra += f"  lr {opt.param_groups[0]['lr']:.2e}"
        print(f"  epoch {ep:2d}  train {tr:.5f}  val {va:.5f}"
              f"{'   ' + extra if extra else ''}"
              f"  [{(time.time() - t0) / 60:.0f} perc]"
              f"{'  *' if improved else ''}")

        if bad >= patience:
            print(f"  early stop ({patience} epoch javulas nelkul)")
            break

    # A LEGJOBB sulyokat adjuk vissza, nem az utolsokat.
    model.load_state_dict(best_state)
    hist["epochs_run"] = len(hist["train"])
    hist["time"] = time.time() - t0

    print(f"  kesz: {hist['ckpt']}  (best val {hist['best']:.5f}, "
          f"{hist['time'] / 60:.1f} perc ebben a futasban)")
    return hist, model


def plot_history(hist, ylabel="loss"):
    """Egy modell tanulasi gorbeje, a learning rate-tel egyutt.

    Az LR a masodik y-tengelyen: igy latszik, hogy a gorbe ellaposodasa a
    konvergencia jele-e, vagy csak az LR csokkent le.
    """
    ep = range(1, len(hist["train"]) + 1)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ep, hist["train"], "-o", ms=3, label="train")
    ax.plot(ep, hist["val"], "-s", ms=3, label="val")
    ax.axhline(hist["best"], ls=":", c="gray",
               label=f"best val {hist['best']:.5f}")
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylabel)
    ax.set_yscale("log")
    ax.grid(alpha=0.3)

    # A regebbi checkpointokban a resume nem toltotte vissza az "lr" listat,
    # igy az rovidebb lehet, mint a train/val. Olyankor a VEGERE igazitjuk:
    # a meglevo ertekek az utolso epochokhoz tartoznak.
    if hist.get("lr"):
        lr = hist["lr"]
        ep_lr = range(len(hist["train"]) - len(lr) + 1, len(hist["train"]) + 1)
        ax2 = ax.twinx()
        ax2.plot(ep_lr, lr, "-", c="tab:green", alpha=0.5, lw=1.2,
                 label="learning rate")
        ax2.set_ylabel("learning rate", color="tab:green")
        ax2.tick_params(axis="y", labelcolor="tab:green")
        ax2.set_yscale("log")
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=8)
    else:
        ax.legend()

    ax.set_title(f"{hist['label']}  ({hist['params'] / 1e6:.1f}M parameter)")
    plt.tight_layout()
    plt.show()

## 4. `bev_ae` — fix BEV statisztika + konvolucios AE

A pontfelho felulnezeti racsra kerul (128x128, 0.8 m cella), cellankent
**harom fix** csatornaval: occupancy, z_max, z_min. Erre megy egy konvolucios
autoencoder.

**Ket dolgot kellett javitani, mindkettot MERES alapjan.**

**1. A loss.** A regi valtozat mind az ot csatornat egyetlen sulyozott MSE-be
tette, es ez elrejtette a valodi problemat: a loss 0.021-en allt, de az
occupancy IoU csak 0.58 volt. Ket oka volt - a z-csatornak ures cellaban
0-t kapnak, a dekoder Sigmoid-ja viszont sosem ad pontos 0-t (le nem vihato
loss-padlo), es az occupancy 0/1 cimkejen az MSE gradiense elhal (a kimeneti
retegen 0.00089 vs BCE-vel 0.0528). Most a magassagok **maszkolva** vannak a
valodi occupancy-val, az occupancy pedig **BCE + Dice** taggal tanul.

**2. A szuk keresztmetszet.** Ez volt a nagyobb nyereseg. A latenst korabban
egy `Linear` par allitotta elo (16384 -> 256, 4.2M parameter), ami a BEV
terbeli szerkezetet egyetlen lapos szorzasba nyomta:

| szukites | IoU | precision | z_mae | param |
|---|---|---|---|---|
| Linear-par | 0.562 | 0.766 | — | 9.8M |
| **conv-neck** | **0.643** | **0.804** | **0.141 m** | **2.8M** |
| nincs bottleneck | 0.895 | 0.941 | — | 1.4M |

A "nincs bottleneck" sor a felso korlat - az a latens 16384 ertek, amit az RL
observation nem tud kezelni. A conv-neck ennek a felso korlatnak a fele utjat
megteszi, harmadannyi parameterbol, mint a Linear-par.

**Csatornak.** A `density` es a `z_mean` kiesett: kimerve az elhagyasuk
gyakorlatilag ingyen van (IoU 0.652 -> 0.643). A `z_mean` a masik ketto
kozott van, a `density`-t pedig az occupancy mar lefedi. A `z_max`+`z_min`
viszont KELL: csak `z_max`-szal a magassaghiba visszaromlik, mert egy
jardaszegely es egy auto nem ugyanaz.


In [ ]:
x_train, x_val = load_clouds(("train", "val"))
print(f"train {tuple(x_train.shape)}  val {tuple(x_val.shape)}  "
      f"{(x_train.numel() + x_val.numel()) * 4 / 1e9:.1f} GB")
model = BEVConvAE()
hist_bev_ae, model = train_model(
    model, label="bev_ae", batch=BATCH["bev_ae"],
    train_data=x_train, val_data=x_val, epochs=EPOCHS, lr=1e-3,
    clip=1.0)
plot_history(hist_bev_ae, ylabel="BCE + Dice + maszkolt MSE")

In [ ]:
free_vram(model)
print("bev_ae utan:", vram())
print("(a pontfelhok a RAM-ban maradnak - a point_mae is azokat hasznalja)")

## 5. `graph_ae` — range image + dinamikus graf

In [ ]:
# A pontfelhoket eldobjuk, hogy a range image-eknek legyen helye.
drop("x_train", "x_val")
r_train, r_val = load_ranges(("train", "val"))
print(f"ranges  train {tuple(r_train.shape)}  val {tuple(r_val.shape)}  "
      f"{(r_train.numel() + r_val.numel()) * 4 / 1e9:.1f} GB")
GRAPH_LR = 4.5e-6

model = LidarAE(DDCONFIG)
hist_graph_ae, model = train_model(
    model, label="graph_ae", batch=BATCH["graph_ae"],
    train_data=r_train, val_data=r_val, epochs=EPOCHS, lr=GRAPH_LR)
plot_history(hist_graph_ae, ylabel="sulyozott L1 (range image)")


In [ ]:
free_vram(model)
print("graph_ae utan:", vram())

## 6. `point_mae` — maszkolt autoencoder

In [ ]:
drop("r_train", "r_val")
if "x_train" not in globals():
    x_train, x_val = load_clouds(("train", "val"))

model = PointMAE(num_group=512)
hist_point_mae, model = train_model(
    model, label="point_mae", batch=40,
    train_data=x_train, val_data=x_val, epochs=EPOCHS, lr=2e-3,
    sched="cosine", resume=True,        
    warmup_frac=0.05)
plot_history(hist_point_mae, ylabel="Chamfer")


In [ ]:
free_vram(model)
print("point_mae utan:", vram())

 # Modell osszehasonlitas


 3 db data sample kivalasztasa es megnezese

In [ ]:
def show_all(pts, raw=None, ri=None, label="", size=(17, 5)):
    pts = pts.cpu().numpy() if hasattr(pts, "cpu") else np.asarray(pts)
    raw = pts if raw is None else (
        raw.cpu().numpy() if hasattr(raw, "cpu") else np.asarray(raw))

    if ri is None:
        ri = points_to_range_image(raw)
    ri = ri.cpu().numpy() if hasattr(ri, "cpu") else np.asarray(ri)
    ri = ri[0] if ri.ndim == 3 else ri

    # A BEV statisztika fix keplet, nem tanult jellemzo - friss peldany is
    # a vegleges racsot adja.
    probe = BEVConvAE().eval()
    with torch.no_grad():
        occ = probe.to_bev(torch.from_numpy(pts).float().unsqueeze(0))[0, 0].numpy()
    r = probe.hparams.pc_range
    free_vram(probe)

    fig, axes = plt.subplots(1, 3, figsize=size)

    # 1. Nyers pontfelho felulnezetbol, magassag szerint szinezve.
    ax = axes[0]
    ax.scatter(pts[:, 0], pts[:, 1], c=pts[:, 2], s=0.3, cmap="viridis",
               vmin=-3, vmax=4, linewidths=0)
    ax.scatter([0], [0], c="red", s=40, marker="^")   # az ego
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]  (elore)")
    ax.set_ylabel("y [m]  (balra)")
    ax.set_title(f"{label}\npontfelho — {len(pts)} pont", fontsize=11)

    # 2. BEV occupancy racs.
    ax = axes[1]
    ax.imshow(occ, cmap="gray_r", origin="lower", extent=[-r, r, -r, r])
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(f"{label}\nBEV occupancy — {100 * occ.mean():.0f}% cella",
                 fontsize=11)

    # 3. Range image.
    ax = axes[2]
    ax.imshow(ri, cmap="magma", aspect="auto", vmin=-1, vmax=1)
    ax.set_xlabel("azimut (512 oszlop)")
    ax.set_ylabel("csatorna (64 sor)")
    ax.set_title(f"{label}\nrange image — "
                 f"{100 * (ri > -0.999).mean():.0f}% kitoltott", fontsize=11)

    plt.tight_layout()
    plt.show()

    # 4. Interaktiv 3D - kulon, mert plotly.
    show_3d(pts, title=f"{label} — 3D pontfelho")


In [ ]:
# --- 3 veletlen frame a datasetbol ----------------------------------------
SHOW_IDX = sorted(np.random.default_rng().choice(len(PATHS), 3, replace=False))

show_raw = [np.load(PATHS[i]) for i in SHOW_IDX]

show_pts = torch.stack([
    torch.from_numpy(a[np.random.default_rng().choice(
        len(a), N_POINTS, replace=len(a) < N_POINTS)].astype(np.float32))
    for a in show_raw])

show_ri = torch.from_numpy(np.stack([points_to_range_image(a) for a in show_raw]))

print(f"framek: {SHOW_IDX}")
print(f"pontfelho {tuple(show_pts.shape)}   range image {tuple(show_ri.shape)}")


### Pelda pontok vizualizacioja

In [ ]:
for f, idx in enumerate(SHOW_IDX):
    show_all(show_pts[f], raw=show_raw[f], ri=show_ri[f],
             label=f"frame {idx}")


# Generaljuk a rekonstrukciokat

In [ ]:
ARCHS = {"BEVConvAE": BEVConvAE, "LidarAE": LidarAE, "PointMAE": PointMAE}
LABELS = ["bev_ae", "graph_ae", "point_mae"]


def range_image_to_points(ri, fov=(-0.9, -25.0),
                          depth_range=(1.0, 50.0), depth_scale=5.68):
    """Range image -> XYZ pontfelho. A `points_to_range_image` INVERZE.

    ri : (1, H, W) vagy (H, W) float, [-1, 1] - a halo kimenete
    return : (K, 3) float32
    """
    ri = ri.detach().cpu().numpy() if hasattr(ri, "detach") else np.asarray(ri)
    ri = ri[0] if ri.ndim == 3 else ri

    # --- 1. denormalizalas: [-1,1] -> log2-skala -> meter ---------------
    d = np.power(2.0, (ri + 1.0) / 2.0 * depth_scale) - 1.0

    # URES CELLAK. A mentes az ures cellat 0 m-re teszi, ami normalizalas utan
    # -1. A dekoder tanh-ja sosem ad pontos -1-et, ezert kuszobbel vagunk.
    keep = d > depth_range[0]
    if not keep.any():
        return np.zeros((0, 3), dtype=np.float32)

    # --- 2. cellacim -> szogek -----------------------------------------
    fov_up, fov_down = fov[0] / 180.0 * np.pi, fov[1] / 180.0 * np.pi
    fov_range = abs(fov_down) + abs(fov_up)

    H, W = ri.shape
    rows, cols = np.nonzero(keep)

    # A cella KOZEPE (+0.5), mert a mentes lefele kerekitett.
    yaw = (2.0 * (cols + 0.5) / W - 1.0) * np.pi                # proj_x inverze
    pitch = (1.0 - (rows + 0.5) / H) * fov_range - abs(fov_down)  # proj_y inverze

    # --- 3. gombi -> Descartes -----------------------------------------
    # A mentesben yaw = -arctan2(y, x) volt, ezert itt a MINUSZ visszajon.
    r = np.clip(d[rows, cols], depth_range[0], depth_range[1])
    cos_p = np.cos(pitch)
    return np.stack([r * cos_p * np.cos(-yaw),
                     r * cos_p * np.sin(-yaw),
                     r * np.sin(pitch)], axis=1).astype(np.float32)


def load_model(label, device=DEVICE, best=True):
    """A mentett modell betoltese.

    best=True : a LEGJOBB val-epoch sulyai (`best_state`), nem az utolsoe.
    """
    ck = torch.load(f"lidar/{label}.ckpt", map_location="cpu", weights_only=False)
    hp = dict(ck.get("hparams") or {})

    if ck["arch"] == "LidarAE":
        model = LidarAE(DDCONFIG, **hp)          # nem Lightning, kell a DDCONFIG
    else:
        model = ARCHS[ck["arch"]](**hp)

    model.load_state_dict((ck.get("best_state") if best else None)
                          or ck["state_dict"])
    return model.to(device).eval(), ck


@torch.no_grad()
def point_mae_reconstruct(model, points):
    """PointMAE -> TELJES rekonstrualt pontfelho.

    A `forward` tanito lepes: maszkol, es csak a maszkolt patch-eket epiti
    vissza. Itt a teljes jelenet kell, ezert tobb menetben futtatjuk ugy, hogy
    minden patch PONTOSAN EGYSZER legyen maszkolt - igy minden pont a modell
    joslata, nem a bemenet masolata.

    points : (1, N, 3) meterben
    return : (G * group_size, 3) float32
    """
    patches, center = model.group(points)      # normalizalt lokalis patch + kozeppont
    tokens = model.embed(patches)
    B, G, C = tokens.shape
    K = model.hparams.group_size

    n_mask = max(1, int(model.hparams.mask_ratio * G))
    n_vis = G - n_mask

    out = torch.zeros(B, G, K, 3, device=points.device)
    filled = torch.zeros(G, dtype=torch.bool, device=points.device)

    for step in range(int(np.ceil(G / n_mask))):
        sel = torch.arange(G, device=points.device).roll(-step * n_mask)[:n_mask]
        mask = torch.zeros(B, G, dtype=torch.bool, device=points.device)
        mask[:, sel] = True

        vis_center = center[~mask].reshape(B, n_vis, 3)
        mask_center = center[mask].reshape(B, n_mask, 3)

        x = tokens[~mask].reshape(B, n_vis, C)
        pos = model.pos_embed(vis_center)
        for blk in model.encoder:
            x = blk(x + pos)
        x = model.encoder_norm(x)

        full = torch.cat([x, model.mask_token.expand(B, n_mask, -1)], dim=1)
        pos_full = model.decoder_pos_embed(
            torch.cat([vis_center, mask_center], dim=1))
        for blk in model.decoder:
            full = blk(full + pos_full)
        rec = model.decoder_norm(full[:, n_vis:])
        rebuild = model.rebuild_head(rec).reshape(B, n_mask, K, 3)

        # Csak amit meg nem toltottunk ki - az utolso menet atfedhet.
        fresh = ~filled[sel]
        out[:, sel[fresh]] = rebuild[:, fresh]
        filled[sel] = True

    # A patch-ek LOKALIS, normalizalt koordinatakban vannak - vissza meterbe.
    pts = model.denorm_patches(out) + model.denorm_centers(center).unsqueeze(2)
    return pts[0].reshape(-1, 3).cpu().numpy().astype(np.float32)


def build_frames(idx_list, device=DEVICE):
    """Frame-indexek -> dict, modellenkenti rekonstrukcioval.

    {"frame 9639": {"raw", "pts", "ri", "bev_ae", "graph_ae", "point_mae"}}

    A modellek EGYESEVEL toltodnek be es szabadulnak fel, igy barmennyi frame
    elfer: a VRAM-ot a modell merete szabja meg, nem a frame-szam.
    """
    # A mar betoltott bemeneteket hasznaljuk, hogy a vizualizalt es a
    # rekonstrualt felho UGYANAZ legyen (a mintavetel seed nelkuli).
    out = {f"frame {i}": {"raw": show_raw[n], "pts": show_pts[n],
                          "ri": show_ri[n]}
           for n, i in enumerate(idx_list)}
    keys = list(out)

    # --- 2. bev_ae: a BEV racs vissza pontokka ------------------------
    model, _ = load_model("bev_ae", device)
    vs, pcr = model.hparams.voxel_size, model.hparams.pc_range
    z_lo_m, z_hi_m = model.hparams.z_min, model.hparams.z_max
    zr = z_hi_m - z_lo_m
    for k in keys:
        with torch.no_grad():
            bev = model(out[k]["pts"].unsqueeze(0).to(device))[0].cpu().numpy()
        occ, zmax_c, zmin_c = bev

        # Ahol az occupancy 0.5 folott van, oda a cella kozeppontjaba ket pont
        # kerul: a z_max es a z_min magassagaba.
        # A to_bev a magassagot (0.05 + 0.95*z)-kent tarolja - hogy az ures
        # cella (0) elvaljon a legalacsonyabb ponttol. Ezt kell visszacsinalni.
        ys, xs = np.nonzero(occ > 0.5)
        cx = (xs + 0.5) * vs - pcr
        cy = (ys + 0.5) * vs - pcr
        z_hi = (zmax_c[ys, xs] - 0.05) / 0.95 * zr + z_lo_m
        z_lo = (zmin_c[ys, xs] - 0.05) / 0.95 * zr + z_lo_m
        out[k]["bev_ae"] = np.concatenate([
            np.stack([cx, cy, z_hi], 1),
            np.stack([cx, cy, z_lo], 1)]).astype(np.float32)
    free_vram(model)

    # --- 3. graph_ae: range image -> range image -> pontok ------------
    model, _ = load_model("graph_ae", device)
    for k in keys:
        with torch.no_grad():
            rec_ri = model(out[k]["ri"].unsqueeze(0).to(device))[0, 0].cpu().numpy()
        out[k]["graph_ae"] = range_image_to_points(rec_ri)
    free_vram(model)

    # --- 4. point_mae: a rekonstrualt patch-ek ------------------------
    model, _ = load_model("point_mae", device)
    for k in keys:
        out[k]["point_mae"] = point_mae_reconstruct(
            model, out[k]["pts"].unsqueeze(0).to(device))
    free_vram(model)

    return out


In [ ]:
FRAMES = build_frames(SHOW_IDX)

for name, d in FRAMES.items():
    print(f"{name:14s} raw {len(d['raw']):6d}  " +
          "  ".join(f"{l} {len(d[l]):5d}" for l in LABELS))


# Vizualizacios osszehansolitasok

In [ ]:
def show_pair(frames, labels=LABELS, size=(13, 5.5), show3d=True):
    from plotly.subplots import make_subplots


    def show_3d_pair(left, right, left_name="eredeti", right_name="rekonstrukcio",
                    title="", max_points=12000, size=1.2, height=600):
        """Ket pontfelho EGYMAS MELLETT, kozos kameraval.

        A ket nezet forgatasa OSSZE VAN KOTVE (shared scene) - amit balra
        elforgatsz, jobbra is fordul, igy ugyanabbol a szogbol latod oket.
        """
        def prep(p):
            p = p.cpu().numpy() if hasattr(p, "cpu") else np.asarray(p)
            if len(p) > max_points:
                p = p[np.random.default_rng(0).choice(len(p), max_points,
                                                    replace=False)]
            return p

        left, right = prep(left), prep(right)

        fig = make_subplots(
            rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
            subplot_titles=(f"{left_name} ({len(left)} pont)",
                            f"{right_name} ({len(right)} pont)"),
            horizontal_spacing=0.02)

        for col, (p, nm) in enumerate([(left, left_name), (right, right_name)], 1):
            fig.add_trace(go.Scatter3d(
                x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers",
                marker=dict(size=size, color=p[:, 2], colorscale="Viridis",
                            cmin=-3, cmax=4, opacity=0.85,
                            showscale=(col == 2),
                            colorbar=dict(title="z [m]", thickness=12, len=0.6)),
            name=nm, showlegend=False,
            hovertemplate="x %{x:.1f}<br>y %{y:.1f}<br>z %{z:.1f}<extra></extra>",
        ), row=1, col=col)

            # Az ego mindket oldalon, hogy legyen kozos viszonyitasi pont.
            fig.add_trace(go.Scatter3d(
                x=[0], y=[0], z=[0], mode="markers",
                marker=dict(size=5, color="red", symbol="diamond"),
                showlegend=False, hoverinfo="skip"), row=1, col=col)

        # aspectmode="data": a valos aranyok. A ket scene kozos kameraja miatt
        # ugyanabbol a szogbol latod oket.
        cam = dict(eye=dict(x=-1.4, y=-1.4, z=0.9))
        scene = dict(aspectmode="data", camera=cam,
                    xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]")
        fig.update_layout(title=title, height=height,
                        margin=dict(l=0, r=0, t=70 if title else 30, b=0),
                        scene=scene, scene2=scene)
        fig.show()

    probe = BEVConvAE().eval()
    r = probe.hparams.pc_range

    def bev_of(pts):
        p = torch.from_numpy(np.asarray(pts)).float()
        with torch.no_grad():
            return probe.to_bev(p.unsqueeze(0))[0, 0].numpy()

    def draw_bev(ax, occ, title):
        ax.imshow(occ, cmap="gray_r", origin="lower", extent=[-r, r, -r, r])
        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")
        ax.set_title(f"{title}\n{100 * occ.mean():.1f}% cella foglalt", fontsize=11)

    def draw_ri(ax, ri, title):
        ri = np.asarray(ri)
        ri = ri[0] if ri.ndim == 3 else ri
        ax.imshow(ri, cmap="magma", aspect="auto", vmin=-1, vmax=1)
        ax.set_xlabel(f"azimut ({ri.shape[1]} oszlop)")
        ax.set_ylabel(f"csatorna ({ri.shape[0]} sor)")
        ax.set_title(f"{title}\n{100 * (ri > -0.999).mean():.0f}% kitoltott",
                     fontsize=11)

    def draw_pts(ax, pts, title):
        pts = np.asarray(pts)
        ax.scatter(pts[:, 0], pts[:, 1], c=pts[:, 2], s=0.3, cmap="viridis",
                   vmin=-3, vmax=4, linewidths=0)
        ax.scatter([0], [0], c="red", s=40, marker="^")
        ax.set_xlim(-50, 50)
        ax.set_ylim(-50, 50)
        ax.set_aspect("equal")
        ax.set_xlabel("x [m]  (elore)")
        ax.set_ylabel("y [m]  (balra)")
        ax.set_title(f"{title}\n{len(pts)} pont", fontsize=11)

    for name, d in frames.items():
        for lbl in labels:
            if lbl not in d:
                continue

            fig, axes = plt.subplots(1, 2, figsize=size)

            if lbl == "bev_ae":
                draw_bev(axes[0], bev_of(d["pts"]), f"{name} — eredeti BEV")
                draw_bev(axes[1], bev_of(d[lbl]), f"{name} — {lbl} rekonstrukcio")
            elif lbl == "graph_ae":
                draw_ri(axes[0], d["ri"], f"{name} — eredeti range image")
                draw_ri(axes[1], points_to_range_image(d[lbl]),
                        f"{name} — {lbl} rekonstrukcio")
            else:
                draw_pts(axes[0], d["pts"], f"{name} — eredeti pontfelho")
                draw_pts(axes[1], d[lbl], f"{name} — {lbl} rekonstrukcio")

            plt.tight_layout()
            plt.show()

            if show3d:
                show_3d_pair(d["pts"], d[lbl],
                             left_name="eredeti", right_name=lbl,
                             title=f"{name} — {lbl}")

    free_vram(probe)


In [ ]:
show_pair(FRAMES)